<a href="https://colab.research.google.com/github/contreras-juan/Material-Ciencia-de-Datos/blob/main/Deep_Learning/Ejercicios/Soluciones/01_Taller_Fundamentos_Redes_Neuronales_Solucion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1 style="color: #FECB05; text-align: center;">Solución — Taller práctico: Fundamentos de redes neuronales</h1>


<h2 style="color: #007ACC;">Autores</h2>

- [Juan Felipe Contreras Alcívar](https://www.linkedin.com/in/juanf-contreras/)

> Esta es una **solución de referencia** del taller [`01_Taller_Fundamentos_Redes_Neuronales.ipynb`](../01_Taller_Fundamentos_Redes_Neuronales.ipynb).
> Los resultados numéricos pueden variar ligeramente según la versión de TensorFlow y la semilla del hardware.


---


<h2 style="color: #007ACC;">Tabla de contenido</h2>

- [Instalación e importaciones](#importaciones)
- [Ejercicio 1. Funciones de activación](#ejercicio-1)
- [Ejercicio 2. Optimizadores](#ejercicio-2)
- [Ejercicio 3. Entrenamiento de una red neuronal](#ejercicio-3)
- [Ejercicio 4. Curvas de entrenamiento](#ejercicio-4)
- [Ejercicio 5. Callbacks](#ejercicio-5)
- [Ejercicio integrador](#ejercicio-integrador)


---

<a id="importaciones"></a>
<h2 style="color: #007ACC;">Instalación e importaciones</h2>

Ejecuta primero la celda de instalación (útil en Google Colab) y luego la de importaciones.


In [ ]:
# Instalación de librerías necesarias para la solución del taller
%pip install -q numpy matplotlib pandas scikit-learn tensorflow

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.datasets import load_wine, make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import SGD, RMSprop, Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, LearningRateScheduler
from tensorflow.keras.regularizers import L2

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)

---

<a id="ejercicio-1"></a>
<h2 style="color: #007ACC;">Ejercicio 1. Funciones de activación</h2>


<h3 style="color: #003366;">1.1 Implementación</h3>


In [ ]:
def relu(x):
    return np.maximum(0, x)

def tanh(x):
    return np.tanh(x)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def softmax(z):
    # Versión numéricamente estable para un vector 1D
    z = np.asarray(z, dtype=float)
    z_shift = z - np.max(z)
    exp_z = np.exp(z_shift)
    return exp_z / np.sum(exp_z)

x = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])
print("ReLU:", relu(x))
print("tanh:", tanh(x))
print("sigmoid:", sigmoid(x))
print("softmax:", softmax(np.array([1.0, 2.0, 3.0])))
print("Suma softmax:", softmax(np.array([1.0, 2.0, 3.0])).sum())

In [ ]:
x_plot = np.linspace(-6, 6, 300)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(x_plot, relu(x_plot), color="C0")
axes[0].set_title("ReLU")
axes[0].grid(True)

axes[1].plot(x_plot, tanh(x_plot), color="C1")
axes[1].set_title("tanh")
axes[1].grid(True)

axes[2].plot(x_plot, sigmoid(x_plot), color="C2")
axes[2].set_title("Sigmoide")
axes[2].grid(True)

for ax in axes:
    ax.axhline(0, color="black", linewidth=0.5)
    ax.axvline(0, color="black", linewidth=0.5)
    ax.set_xlabel("x")

plt.tight_layout()
plt.show()

<h3 style="color: #003366;">1.2 Preguntas de reflexión — respuestas</h3>

1. **Activaciones lineales:** la composición de transformaciones afines es otra transformación afín. Sin no linealidad, varias capas `Dense` se colapsan en una sola capa lineal (o logística si la salida es sigmoide/softmax). Por eso las activaciones no lineales son esenciales para aprender patrones complejos.

2. **Softmax:** se usa en la **capa de salida** de clasificación **multiclase**, porque convierte logits en un vector de probabilidades que suma 1.

3. **ReLU vs sigmoide:** ReLU es más barata de calcular, mitiga mejor la saturación/vanishing gradients en capas ocultas profundas y favorece representaciones dispersas (muchas activaciones en cero).

4. **Teorema de Aproximación Universal:** garantiza que una red con una capa oculta y activación no lineal adecuada *puede* aproximar cualquier función continua en un compacto con error arbitrario. **No** garantiza que el entrenamiento encuentre esos pesos, ni da un tamaño práctico de red, ni evita sobreajuste.


---

<a id="ejercicio-2"></a>
<h2 style="color: #007ACC;">Ejercicio 2. Optimizadores</h2>


<h3 style="color: #003366;">2.1 Descenso de gradiente unidimensional</h3>


In [ ]:
def f(x):
    return x**4 - 3 * x**3 + 2

def grad_f(x):
    return 4 * x**3 - 9 * x**2

def gradient_descent(x0, lr, n_iters):
    path = [x0]
    x = x0
    for _ in range(n_iters):
        x = x - lr * grad_f(x)
        path.append(x)
    return path

x_values = np.linspace(-1, 3.5, 400)
plt.figure(figsize=(10, 6))
plt.plot(x_values, f(x_values), label=r"$f(x)=x^4-3x^3+2$", color="black")

for lr, color in [(0.001, "C0"), (0.01, "C1"), (0.05, "C3")]:
    path = gradient_descent(x0=-0.5, lr=lr, n_iters=30)
    plt.plot(path, [f(x) for x in path], "o-", markersize=4, label=rf"$\eta={lr}$", color=color)

plt.title("Trayectorias del descenso de gradiente")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.legend()
plt.grid(True)
plt.show()

print("Comentario: eta muy pequeña avanza lento; eta moderada desciende bien;")
print("eta grande puede oscilar o volverse inestable cerca de regiones con gradiente pronunciado.")

<h3 style="color: #003366;">2.2 Comparación de optimizadores</h3>


In [ ]:
X_moons, y_moons = make_moons(n_samples=1000, noise=0.25, random_state=SEED)
X_moons = StandardScaler().fit_transform(X_moons)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_moons, y_moons, test_size=0.2, random_state=SEED, stratify=y_moons
)

def build_binary_model():
    return Sequential([
        Dense(16, activation="relu", input_shape=(2,)),
        Dense(8, activation="relu"),
        Dense(1, activation="sigmoid"),
    ])

optimizers = {
    "SGD": SGD(learning_rate=0.05),
    "RMSprop": RMSprop(learning_rate=0.01),
    "Adam": Adam(learning_rate=0.01),
}

histories = {}

for name, opt in optimizers.items():
    tf.random.set_seed(SEED)
    model = build_binary_model()
    model.compile(optimizer=opt, loss="binary_crossentropy", metrics=["accuracy"])
    history = model.fit(
        X_tr, y_tr,
        validation_split=0.2,
        epochs=40,
        batch_size=32,
        verbose=0,
    )
    histories[name] = history
    test_loss, test_acc = model.evaluate(X_te, y_te, verbose=0)
    print(f"{name:8s} | test loss={test_loss:.4f} | test acc={test_acc:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, history in histories.items():
    axes[0].plot(history.history["loss"], label=f"{name} train")
    axes[0].plot(history.history["val_loss"], "--", label=f"{name} val")
    axes[1].plot(history.history["accuracy"], label=f"{name} train")
    axes[1].plot(history.history["val_accuracy"], "--", label=f"{name} val")

axes[0].set_title("Pérdida")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("loss")
axes[0].legend()
axes[0].grid(True)

axes[1].set_title("Exactitud")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("accuracy")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

**Respuesta (típica):** Adam y RMSprop suelen converger más rápido y con curvas más suaves que SGD. SGD, incluso con una tasa relativamente alta, tiende a bajar más lento y puede mostrar más oscilaciones en `val_loss`/`val_accuracy`, porque no adapta la tasa por parámetro ni acumula momentum adaptativo como Adam.


---

<a id="ejercicio-3"></a>
<h2 style="color: #007ACC;">Ejercicio 3. Entrenamiento de una red neuronal</h2>


<h3 style="color: #003366;">3.1 Clasificación con Wine</h3>


In [ ]:
wine = load_wine()
X = wine.data
y = wine.target.reshape(-1, 1)
class_names = wine.target_names

try:
    encoder = OneHotEncoder(sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(sparse=False)

y_oh = encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_oh, test_size=0.2, random_state=SEED, stratify=y_oh
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

tf.random.set_seed(SEED)
model_wine = Sequential([
    Dense(16, activation="relu", input_shape=(X_train_s.shape[1],)),
    Dense(8, activation="relu"),
    Dense(3, activation="softmax"),
])

model_wine.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

history_wine = model_wine.fit(
    X_train_s, y_train,
    epochs=80,
    batch_size=16,
    validation_split=0.2,
    verbose=0,
)

test_loss, test_acc = model_wine.evaluate(X_test_s, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

y_pred = np.argmax(model_wine.predict(X_test_s, verbose=0), axis=1)
y_true = np.argmax(y_test, axis=1)

print("\nMatriz de confusión:")
print(confusion_matrix(y_true, y_pred))
print("\nClassification report:")
print(classification_report(y_true, y_pred, target_names=class_names))

<h3 style="color: #003366;">3.2 Preguntas — respuestas</h3>

1. Hay **3 clases mutuamente excluyentes**. Softmax produce una distribución de probabilidad conjunta (suma 1). Sigmoide independiente por neurona sería más propia de clasificación **multi-etiqueta**.

2. Sin estandarizar, variables con escalas muy distintas dominan el gradiente. El entrenamiento se vuelve más lento o inestable y la convergencia empeora.

3. En este dataset pequeño y bien separado, la exactitud en test suele ser alta. Si alguna clase concentra más errores, suele ser por solapamiento de características; revisa la matriz de confusión impresa arriba.


---

<a id="ejercicio-4"></a>
<h2 style="color: #007ACC;">Ejercicio 4. Curvas de entrenamiento</h2>


<h3 style="color: #003366;">4.1 Modelo propenso a sobreajuste</h3>


In [ ]:
def plot_training_curves(history, metrics=("loss", "accuracy"), title_prefix=""):
    history_dict = history.history if hasattr(history, "history") else history
    n = len(metrics)
    plt.figure(figsize=(6 * n, 4))
    for i, metric in enumerate(metrics):
        plt.subplot(1, n, i + 1)
        plt.plot(history_dict[metric], label=f"train {metric}")
        plt.plot(history_dict[f"val_{metric}"], label=f"val {metric}")
        plt.xlabel("Epochs")
        plt.ylabel(metric)
        plt.title(f"{title_prefix}{metric}")
        plt.legend()
        plt.grid(True)
    plt.tight_layout()
    plt.show()

tf.random.set_seed(SEED)
model_over = Sequential([
    Dense(64, activation="relu", input_shape=(X_train_s.shape[1],)),
    Dense(64, activation="relu"),
    Dense(32, activation="relu"),
    Dense(3, activation="softmax"),
])

model_over.compile(optimizer=Adam(learning_rate=0.01), loss="categorical_crossentropy", metrics=["accuracy"])

history_over = model_over.fit(
    X_train_s, y_train,
    validation_split=0.2,
    epochs=150,
    batch_size=16,
    verbose=0,
)

plot_training_curves(history_over, title_prefix="Sobreajuste potencial — ")
print("Épocas entrenadas:", len(history_over.history["loss"]))
print("Train acc final:", history_over.history["accuracy"][-1])
print("Val acc final:", history_over.history["val_accuracy"][-1])

<h3 style="color: #003366;">4.2 Diagnóstico y corrección</h3>

**Diagnóstico esperado:** si `train loss` sigue bajando mientras `val loss` se estanca o sube (y/o hay una brecha grande en accuracy), el patrón es de **sobreajuste**. Wine es pequeño, así que un modelo 64-64-32 con muchas épocas es un buen candidato a memorizar.

**Corrección aplicada:** red más pequeña + `Dropout` + regularización `L2`.


In [ ]:
tf.random.set_seed(SEED)
model_reg = Sequential([
    Dense(16, activation="relu", input_shape=(X_train_s.shape[1],), kernel_regularizer=L2(1e-3)),
    Dropout(0.3),
    Dense(8, activation="relu", kernel_regularizer=L2(1e-3)),
    Dropout(0.2),
    Dense(3, activation="softmax"),
])

model_reg.compile(optimizer=Adam(learning_rate=0.005), loss="categorical_crossentropy", metrics=["accuracy"])

history_reg = model_reg.fit(
    X_train_s, y_train,
    validation_split=0.2,
    epochs=150,
    batch_size=16,
    verbose=0,
)

plot_training_curves(history_reg, title_prefix="Modelo regularizado — ")

_, acc_over = model_over.evaluate(X_test_s, y_test, verbose=0)
_, acc_reg = model_reg.evaluate(X_test_s, y_test, verbose=0)
print(f"Test accuracy modelo grande: {acc_over:.4f}")
print(f"Test accuracy modelo regularizado: {acc_reg:.4f}")

**Respuesta:** la regularización suele **cerrar la brecha** train/val. Es normal sacrificar un poco de exactitud de entrenamiento a cambio de curvas más alineadas y, con suerte, mejor o similar generalización en test.


---

<a id="ejercicio-5"></a>
<h2 style="color: #007ACC;">Ejercicio 5. Callbacks</h2>


<h3 style="color: #003366;">5.1 EarlyStopping + ReduceLROnPlateau</h3>


In [ ]:
tf.random.set_seed(SEED)
model_cb = Sequential([
    Dense(32, activation="relu", input_shape=(X_train_s.shape[1],)),
    Dense(16, activation="relu"),
    Dense(3, activation="softmax"),
])

model_cb.compile(optimizer=Adam(learning_rate=0.01), loss="categorical_crossentropy", metrics=["accuracy"])

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=12,
    restore_best_weights=True,
    verbose=1,
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1,
)

history_cb = model_cb.fit(
    X_train_s, y_train,
    validation_split=0.2,
    epochs=200,
    batch_size=16,
    callbacks=[early_stopping, reduce_lr],
    verbose=0,
)

n_epochs = len(history_cb.history["loss"])
print(f"Entrenamiento detenido tras {n_epochs} épocas (máximo pedido: 200).")
if "lr" in history_cb.history:
    print("LR inicial:", history_cb.history["lr"][0])
    print("LR final:", history_cb.history["lr"][-1])
else:
    print("Nota: la clave 'lr' puede no aparecer en history según la versión de Keras;")
    print("revisa los logs de ReduceLROnPlateau para ver las reducciones.")

plot_training_curves(history_cb, title_prefix="Callbacks — ")
test_loss_cb, test_acc_cb = model_cb.evaluate(X_test_s, y_test, verbose=0)
print(f"Test loss={test_loss_cb:.4f} | Test acc={test_acc_cb:.4f}")

<h3 style="color: #003366;">5.2 LearningRateScheduler (step decay)</h3>


In [ ]:
def lr_schedule(epoch, lr=None):
    initial_lr = 0.01
    return initial_lr * (0.5 ** (epoch // 10))

lr_scheduler = LearningRateScheduler(lr_schedule, verbose=0)

tf.random.set_seed(SEED)
model_lrs = Sequential([
    Dense(16, activation="relu", input_shape=(X_train_s.shape[1],)),
    Dense(8, activation="relu"),
    Dense(3, activation="softmax"),
])

model_lrs.compile(
    optimizer=Adam(learning_rate=0.01),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

history_lrs = model_lrs.fit(
    X_train_s, y_train,
    epochs=40,
    batch_size=16,
    validation_split=0.2,
    callbacks=[lr_scheduler],
    verbose=0,
)

epochs = np.arange(40)
lrs = [lr_schedule(e) for e in epochs]

plt.figure(figsize=(8, 4))
plt.plot(epochs, lrs, "o-", color="C0")
plt.title(r"Step decay: $\eta_t = 0.01 \cdot 0.5^{\lfloor t/10 \rfloor}$")
plt.xlabel("Epoch")
plt.ylabel("Learning rate")
plt.grid(True)
plt.show()

plot_training_curves(history_lrs, title_prefix="LR Scheduler — ")

---

<a id="ejercicio-integrador"></a>
<h2 style="color: #007ACC;">Ejercicio integrador</h2>

Comparamos una arquitectura **simple** frente a una **más profunda/ancha** sobre Wine, ambas con Adam + EarlyStopping + ReduceLROnPlateau.


In [ ]:
def make_callbacks():
    return [
        EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=6, min_lr=1e-6, verbose=0),
    ]

architectures = {
    "simple": lambda: Sequential([
        Dense(8, activation="relu", input_shape=(X_train_s.shape[1],)),
        Dense(3, activation="softmax"),
    ]),
    "profunda": lambda: Sequential([
        Dense(64, activation="relu", input_shape=(X_train_s.shape[1],)),
        Dense(32, activation="relu"),
        Dense(16, activation="relu"),
        Dense(3, activation="softmax"),
    ]),
}

results = []
histories_int = {}

for name, builder in architectures.items():
    tf.random.set_seed(SEED)
    model = builder()
    model.compile(optimizer=Adam(learning_rate=0.01), loss="categorical_crossentropy", metrics=["accuracy"])

    history = model.fit(
        X_train_s, y_train,
        validation_split=0.2,
        epochs=200,
        batch_size=16,
        callbacks=make_callbacks(),
        verbose=0,
    )
    histories_int[name] = history

    test_loss, test_acc = model.evaluate(X_test_s, y_test, verbose=0)
    train_acc = history.history["accuracy"][-1]
    val_acc = history.history["val_accuracy"][-1]
    n_epochs = len(history.history["loss"])

    results.append({
        "arquitectura": name,
        "epochs_efectivas": n_epochs,
        "train_acc_final": round(train_acc, 4),
        "val_acc_final": round(val_acc, 4),
        "test_acc": round(test_acc, 4),
        "test_loss": round(test_loss, 4),
        "gap_train_val": round(train_acc - val_acc, 4),
    })

    plot_training_curves(history, title_prefix=f"{name} — ")

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

best = df_results.sort_values(["test_acc", "gap_train_val"], ascending=[False, True]).iloc[0]
print("\nGanador sugerido:", best["arquitectura"])
print(
    "Conclusión: preferimos la arquitectura con mejor desempeño en test y menor evidencia de sobreajuste "
    f"(gap train/val = {best['gap_train_val']}). En datasets pequeños como Wine, un modelo simple a menudo "
    "generaliza igual o mejor que uno profundo si este último memoriza el entrenamiento."
)

---

<h2 style="color: #007ACC;">Notas para el docente</h2>

- Wine tiene pocas observaciones: las métricas de test pueden variar entre ejecuciones.
- Si `OneHotEncoder` falla por versión de scikit-learn, el notebook ya contempla el fallback `sparse=False`.
- Se mantiene el estilo de los cuadernos del curso (`Sequential` + `input_shape` en la primera capa).
